# Build Fact Orders
1. read the data from the silver orders table
2. convert timestamp columns into date columns and add delivery_days, estimated_delivery_days and delivery_status
3. join silver orders table, dim_customers table and dim_date table
4. select the required columns
- Measures: delivery_days, estimated_delivery_days
- Foreign keys: customer_sk,order_purchase_date_key, order_approved_date_key, order_delivered_carrier_date_key, order_delivered_customer_date_key, order_estimated_delivery_date_key
- Degenerate Dimensions: order_id, order_status
5. write the transformed data to gold fact_orders table

### Step1 - read the data from the silver orders table

In [0]:
#Import
from pyspark.sql.functions import col,cast,date_diff,when

In [0]:
orders_df = spark.read.table("olist_catalog.silver.orders")

### Step2 - convert timestamp columns into date columns and add delivery_days, estimated_delivery_days and delivery_status

In [0]:
orders_date_format_df = (
    orders_df.select(
        col("order_id"),
        col("customer_id"),
        col("order_status"),
        col("order_purchase_timestamp").cast("date").alias("order_purchase_date"),
        col("order_approved_at").cast("date").alias("order_approved_date"),
        col("order_delivered_carrier_date").cast("date"),
        col("order_delivered_customer_date").cast("date"),
        col("order_estimated_delivery_date").cast("date"),
        date_diff(col("order_delivered_customer_date"),col("order_purchase_date")).alias("delivery_days"),
        date_diff(col("order_estimated_delivery_date"),col("order_purchase_date")).alias("estimated_delivery_days")
    )
)


In [0]:
orders_date_format_df = orders_date_format_df.withColumn(
    "delivery_status",
    when(col("order_status") != "delivered", "not_delivered")
    .when(col("delivery_days") <= col("estimated_delivery_days"), "on_time")
    .otherwise("late")
)

### Step3 - join silver orders table, dim_customers table and dim_date table

In [0]:
dim_customers_df = spark.read.table("olist_catalog.gold.dim_customers")

dim_date_df = spark.read.table("olist_catalog.gold.dim_date")

In [0]:
fact_orders_df = (
    orders_date_format_df.alias("o")
        .join(
            dim_customers_df.alias("c"),
            col("o.customer_id")==col("c.customer_id"),
            "left"
        )
        .join(
            dim_date_df.alias("p"),
            col("o.order_purchase_date")==col("p.full_date"),
            "left"
        )
        .join(
            dim_date_df.alias("a"),
            col("o.order_approved_date")==col("a.full_date"),
            "left"
        )
        .join(
            dim_date_df.alias("cr"),
            col("o.order_delivered_carrier_date")==col("cr.full_date"),
            "left"
        )
        .join(
            dim_date_df.alias("cm"),
            col("o.order_delivered_customer_date")==col("cm.full_date"),
            "left"
        )
        .join(
            dim_date_df.alias("e"),
            col("o.order_estimated_delivery_date")==col("e.full_date"),
            "left"
        )
)

### Step4 - select the required columns

In [0]:
fact_orders_final_df = fact_orders_df.select(
    col("o.order_id"),
    col("o.order_status"),
    col("c.customer_sk"),
    col("p.date_key").alias("order_purchase_date_key"),
    col("a.date_key").alias("order_approved_date_key"),
    col("cr.date_key").alias("order_delivered_carrier_date_key"),
    col("cm.date_key").alias("order_delivered_customer_date_key"),
    col("e.date_key").alias("order_estimated_delivery_date_key"),
    col("o.delivery_days"),
    col("o.estimated_delivery_days"),
    col("delivery_status")
)

### Step5 - write the transformed data to gold fact_orders table

In [0]:
(
    fact_orders_final_df.write
        .format("delta")
        .option("overwriteSchema","True")
        .mode("overwrite")
        .saveAsTable("olist_catalog.gold.fact_orders")
)

In [0]:
%sql
select * from olist_catalog.gold.fact_orders